# Practice 5 — Dynamic Web Scraping & Data Cleaning

**Source website:** https://webscraper.io/test-sites/e-commerce/more  


In [4]:
!pip install -q pandera playwright nest_asyncio
!playwright install --with-deps chromium

Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [79.0 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://cli.github.com/packages stable InRelease [3,917 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Pa

In [8]:
import pandas as pd
import pandera.pandas as pa
from playwright.async_api import async_playwright

## Task 1. Dynamic data collection

In [11]:
%%writefile scrape.py
import pandas as pd
from playwright.sync_api import sync_playwright

pages = {
    "laptops": "https://webscraper.io/test-sites/e-commerce/more/computers/laptops",
    "tablets": "https://webscraper.io/test-sites/e-commerce/more/computers/tablets",
}

def get_value(card, selector, attribute=None):
    element = card.locator(selector)
    if element.count() == 0:
        return None
    if attribute:
        return element.first.get_attribute(attribute)
    return element.first.inner_text()

rows = []
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    for category, url in pages.items():
        page.goto(url)

        if page.locator(".acceptCookies").first.is_visible():
            page.locator(".acceptCookies").first.click()


        while page.locator(".ecomerce-items-scroll-more").is_visible():
            page.click(".ecomerce-items-scroll-more")
            page.wait_for_timeout(1000)

        for card in page.locator(".product-wrapper").all():
            rows.append({
                "title": get_value(card, "a.title", "title"),
                "price": get_value(card, ".price"),
                "description": get_value(card, ".description"),
                "reviews": get_value(card, ".review-count"),
                "rating": get_value(card, "[data-rating]", "data-rating"),
                "category": category,
            })

    browser.close()

pd.DataFrame(rows).to_csv("scraped_raw.csv", index=False)
print("Done:", len(rows), "rows")

Writing scrape.py


## Task 2. Missing values

In [13]:
df = df_raw.copy()

# empty text also counts as missing
df = df.replace("", None)

print("Missing values BEFORE:")
print(df.isna().sum())

Missing values BEFORE:
title            0
price            0
description      0
reviews          0
rating         138
category         0
dtype: int64


In [14]:
# Rules:
# - no title or no price -> drop the row (useless record)
# - no description       -> fill with "no description"
# - no reviews           -> fill with "0 reviews"
# - no rating            -> keep as missing (we do not want to invent a rating)
df = df.dropna(subset=["title", "price"])
df["description"] = df["description"].fillna("no description")
df["reviews"] = df["reviews"].fillna("0 reviews")

print("Missing values AFTER:")
print(df.isna().sum())

Missing values AFTER:
title            0
price            0
description      0
reviews          0
rating         138
category         0
dtype: int64


## Task 3. String cleaning

In [15]:
# 1. remove extra spaces and make titles look the same
df["title"] = df["title"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
df["description"] = df["description"].str.strip().str.replace(r"\s+", " ", regex=True)

# 2. price: "$1,139.54" -> 1139.54
df["price"] = df["price"].str.replace(r"[$,]", "", regex=True).astype(float)

# 3. reviews: "14 reviews" -> 14
df["reviews"] = df["reviews"].str.extract(r"(\d+)")[0].astype(int)

# 4. rating: text "3" -> number 3
df["rating"] = pd.to_numeric(df["rating"]).astype(float)

# 5. brand = first word of the title
df["brand"] = df["title"].str.split().str[0]

df.head()

,title,price,description,reviews,rating,category,brand
0,Asus Vivobook X441Na-Ga190,295.99,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",1,NaN,laptops,Asus
1,Prestigio Smartbook 133S Dark Grey,299.00,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",9,NaN,laptops,Prestigio
2,Prestigio Smartbook 133S Gold,299.00,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ...",9,NaN,laptops,Prestigio
3,Aspire E1-510,306.99,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",2,NaN,laptops,Aspire
4,Lenovo V110-15Iap,321.94,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1...",12,NaN,laptops,Lenovo


## Task 4. Data validation with Pandera

In [16]:
schema = pa.DataFrameSchema({
    "title":    pa.Column(str, nullable=False),
    "price":    pa.Column(float, pa.Check.between(1, 5000)),          # allowed range
    "reviews":  pa.Column(int, pa.Check.ge(0)),                       # not negative
    "rating":   pa.Column(float, pa.Check.between(1, 5), nullable=True),
    "category": pa.Column(str, pa.Check.isin(["laptops", "tablets"])),  # allowed values
    "brand":    pa.Column(str, nullable=False),
})

try:
    schema.validate(df, lazy=True)
    print("Validation OK, no problems found")
except pa.errors.SchemaErrors as e:
    print("Problems found:")
    print(e.failure_cases)

Validation OK, no problems found


In [17]:
# If problems were found above, fix them here. Example: remove rows with a bad price.
df = df[df["price"].between(1, 5000)]

schema.validate(df)
print("Validation OK after fixing")

Validation OK after fixing


## Task 5. Final dataset

In [18]:
print("Duplicates:", df.duplicated().sum())
df_final = df.drop_duplicates().reset_index(drop=True)

print("\nMissing values:")
print(df_final.isna().sum())
print("\nData types:")
print(df_final.dtypes)

schema.validate(df_final)
df_final.to_csv("final_dataset.csv", index=False)
df_final.head()

Duplicates: 0

Missing values:
title            0
price            0
description      0
reviews          0
rating         138
category         0
brand            0
dtype: int64

Data types:
title           object
price          float64
description     object
reviews          int64
rating         float64
category        object
brand           object
dtype: object


,title,price,description,reviews,rating,category,brand
0,Asus Vivobook X441Na-Ga190,295.99,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",1,NaN,laptops,Asus
1,Prestigio Smartbook 133S Dark Grey,299.00,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",9,NaN,laptops,Prestigio
2,Prestigio Smartbook 133S Gold,299.00,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ...",9,NaN,laptops,Prestigio
3,Aspire E1-510,306.99,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",2,NaN,laptops,Aspire
4,Lenovo V110-15Iap,321.94,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1...",12,NaN,laptops,Lenovo


In [19]:
# Raw vs cleaned
print("Rows: raw =", len(df_raw), " clean =", len(df_final))
print("Raw types:  ", dict(df_raw.dtypes.astype(str)))
print("Clean types:", dict(df_final.dtypes.astype(str)))
df_raw[["price", "reviews"]].head(3).join(df_final[["price", "reviews"]].head(3), rsuffix="_clean")

Rows: raw = 138  clean = 138
Raw types:   {'title': 'object', 'price': 'object', 'description': 'object', 'reviews': 'object', 'rating': 'object', 'category': 'object'}
Clean types: {'title': 'object', 'price': 'float64', 'description': 'object', 'reviews': 'int64', 'rating': 'float64', 'category': 'object', 'brand': 'object'}


,price,reviews,price_clean,reviews_clean
0,$295.99,1 reviews,295.99,1
1,$299,9 reviews,299.00,9
2,$299,9 reviews,299.00,9


## Five findings
*(Replace the numbers with your own results.)*

1. **Price was text** (`"$1,139.54"`). The `$` and `,` were removed with regex and the column became `float`.
2. **Reviews were text** (`"14 reviews"`). The number was extracted with regex and the column became `int`.
3. **Missing values:** X before and Y after cleaning. Rows without title or price were dropped; missing rating was left empty.
4. **Extra spaces and different text format** in titles and descriptions were fixed; a new `brand` column was created from the title.
5. **Validation:** Pandera checked types, price range, rating 1–5 and allowed categories. D duplicate rows were removed; the final dataset has R rows and passes the schema.